# Name Entity Recognition using Deep Learning

Local version of the notebook for this repository.

Expected layout from the notebook working directory:

* `NERC-nn/`
* `lab_resources/DDI/data/{train,devel,test}`
* `lab_resources/DDI/util/evaluator.py`
* `requirements.txt`

Before running the notebook locally:

* create and activate a Python virtual environment
* install the dependencies with `%pip install -r requirements.txt`
* open the notebook from the repository root so the relative paths below resolve correctly


In [57]:
# %pip install -r requirements.txt

In [58]:
from pathlib import Path

rootdir = Path.cwd()
rootdir

WindowsPath('d:/cosas_uni/master/Q2_25-26/MUD/NERC_task')

Define the local paths to the code, data and evaluator:

In [59]:
utilsdir = str(rootdir / 'NERC-nn')
evaluatordir = str(rootdir / 'lab_resources' / 'DDI' / 'util')
traindir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'train')
validationdir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'devel')
testdir = str(rootdir / 'lab_resources' / 'DDI' / 'data' / 'test')
modelname = str(rootdir / 'model.keras')
outfile = str(rootdir / 'out.txt')

for path in [utilsdir, evaluatordir, traindir, validationdir, testdir]:
    print(path, '->', Path(path).exists())

d:\cosas_uni\master\Q2_25-26\MUD\NERC_task\NERC-nn -> True
d:\cosas_uni\master\Q2_25-26\MUD\NERC_task\lab_resources\DDI\util -> True
d:\cosas_uni\master\Q2_25-26\MUD\NERC_task\lab_resources\DDI\data\train -> True
d:\cosas_uni\master\Q2_25-26\MUD\NERC_task\lab_resources\DDI\data\devel -> True
d:\cosas_uni\master\Q2_25-26\MUD\NERC_task\lab_resources\DDI\data\test -> True


In [60]:
import sys
sys.path.insert(1, str(utilsdir))
sys.path.insert(1, str(evaluatordir))


In [61]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

from contextlib import redirect_stdout

from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, GRU, Embedding, Dense, TimeDistributed, Dropout, Bidirectional, concatenate, Softmax
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import codemaps

import codemaps_baseline
import biobert_ner

import nltk
nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pablo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def build_network(codes, recurrent_size) :

   # sizes
   n_words = codes.get_n_words()
   n_lc_words = codes.get_n_lc_words()
   n_sufs = codes.get_n_sufs()
   n_shapes = codes.get_n_shapes()
   n_caps = codes.get_n_caps()
   n_nums = codes.get_n_nums()
   n_dashes = codes.get_n_dashes()
   n_gazetteer = codes.get_n_gazetteer()
   n_labels = codes.get_n_labels()   
   max_len = codes.maxlen

   inptW = Input(shape=(max_len,)) # word input layer & embeddings
   # Use the main word sequence as the single source of truth for masking.
   embW = Embedding(input_dim=n_words, output_dim=100,
                    input_length=max_len, mask_zero=True)(inptW)  

   inptLW = Input(shape=(max_len,)) # lowercased word input layer & embeddings
   embLW = Embedding(input_dim=n_lc_words, output_dim=100,
                     input_length=max_len, mask_zero=False)(inptLW)
   
   inptS = Input(shape=(max_len,))  # suf input layer & embeddings
   embS = Embedding(input_dim=n_sufs, output_dim=50,
                    input_length=max_len, mask_zero=False)(inptS) 

   inptC = Input(shape=(max_len,))  # capitalization feature
   embC = Embedding(input_dim=n_caps, output_dim=8,
                    input_length=max_len, mask_zero=False)(inptC)

   inptN = Input(shape=(max_len,))  # number feature
   embN = Embedding(input_dim=n_nums, output_dim=6,
                    input_length=max_len, mask_zero=False)(inptN)

   inptD = Input(shape=(max_len,))  # dash feature
   embD = Embedding(input_dim=n_dashes, output_dim=4,
                    input_length=max_len, mask_zero=False)(inptD)

   inptSh = Input(shape=(max_len,))  # token shape feature
   embSh = Embedding(input_dim=n_shapes, output_dim=20,
                     input_length=max_len, mask_zero=False)(inptSh)

   inptG = Input(shape=(max_len,))  # gazetteer feature
   embG = Embedding(input_dim=n_gazetteer, output_dim=8,
                    input_length=max_len, mask_zero=False)(inptG)

   dropW = Dropout(0.1)(embW)
   dropLW = Dropout(0.1)(embLW)
   dropS = Dropout(0.1)(embS)
   dropC = Dropout(0.05)(embC)
   dropN = Dropout(0.05)(embN)
   dropD = Dropout(0.05)(embD)
   dropSh = Dropout(0.1)(embSh)
   dropG = Dropout(0.05)(embG)
   drops = concatenate([dropW, dropLW, dropS, dropC, dropN, dropD, dropSh, dropG])

   # biLSTM   
   bilstm = Bidirectional(LSTM(units=recurrent_size, return_sequences=True,
                               dropout=0.1, recurrent_dropout=0.0))(drops) 
   # output softmax layer
   out = TimeDistributed(Dense(n_labels, activation="softmax"))(bilstm)

   # build and compile model
   model = Model([inptW, inptLW, inptS, inptC, inptN, inptD, inptSh, inptG], out)
   model.compile(optimizer="adam",
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
   
   return model



In [63]:
# directory with files to process


# load train and validation data
traindata = codemaps.Dataset(traindir)

valdata = codemaps.Dataset(validationdir)

# create indexes from training data
max_len = 200
suf_len = 5
codes = codemaps.Codemaps(traindata, max_len, suf_len)

# encode datasets
# 1st ver -> [Xt,Xts] = codes.encode_words(traindata)
Xt = codes.encode_words(traindata)
Yt = codes.encode_labels(traindata)

# 1st ver -> [Xv,Xvs] = codes.encode_words(valdata)
Xv = codes.encode_words(valdata)
Yv = codes.encode_labels(valdata)

n_tags = codes.get_n_labels()
max_len = codes.maxlen

In [64]:
lr = 5e-4
recurrent_size = 200
model = build_network2(codes, recurrent_size)
model.compile(optimizer=Adam(learning_rate = lr) ,metrics=["accuracy"], loss="sparse_categorical_crossentropy")

with redirect_stdout(sys.stderr) :
   model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_32      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_33      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_34      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_35      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_36      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_37      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_38      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_39      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_32        │ (None, 200, 100)  │    967,600 │ input_layer_32[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_33        │ (None, 200, 100)  │    833,900 │ input_layer_33[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_34        │ (None, 200, 50)   │    247,850 │ input_layer_34[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_35        │ (None, 200, 8)    │         48 │ input_layer_35[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_36        │ (None, 200, 6)    │         24 │ input_layer_36[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_37        │ (None, 200, 4)    │         12 │ input_layer_37[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_38        │ (None, 200, 20)   │      6,000 │ input_layer_38[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_39        │ (None, 200, 8)    │         80 │ input_layer_39[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_32          │ (None, 200, 100)  │          0 │ embedding_32[0][

 Total params: 2,854,724 (10.89 MB)

 Trainable params: 2,854,724 (10.89 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  train.py ../data/Train ../data/Devel  modelname
## --

# train model
n_epochs = 25
batch_size = 16
callbacks = [
   EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]
with redirect_stdout(sys.stderr):
   model.fit(Xt, Yt, batch_size=batch_size, epochs=n_epochs, validation_data=(Xv, Yv), verbose=1, callbacks=callbacks)
   
   # baseline
   # model.fit(Xt, Yt, batch_size=32, epochs=n_epochs, validation_data=(Xv, Yv), verbose=1)

# save model and indexs
model.save(modelname)
codes.save(modelname)
#save_model_and_indexs(model, idx, modelname)

Epoch 1/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 71s 193ms/step - accuracy: 0.9206 - loss: 0.3561 - val_accuracy: 0.9652 - val_loss: 0.1375 - learning_rate: 5.0000e-04
Epoch 2/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 61s 179ms/step - accuracy: 0.9763 - loss: 0.0857 - val_accuracy: 0.9753 - val_loss: 0.0976 - learning_rate: 5.0000e-04
Epoch 3/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 66s 194ms/step - accuracy: 0.9868 - loss: 0.0477 - val_accuracy: 0.9797 - val_loss: 0.0838 - learning_rate: 5.0000e-04
Epoch 4/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 67s 199ms/step - accuracy: 0.9919 - loss: 0.0306 - val_accuracy: 0.9789 - val_loss: 0.0856 - learning_rate: 5.0000e-04
Epoch 5/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 67s 196ms/step - accuracy: 0.9944 - loss: 0.0208 - val_accuracy: 0.9792 - val_loss: 0.0875 - learning_rate: 5.0000e-04
Epoch 6/25
339/339 ━━━━━━━━━━━━━━━━━━━━ 70s 206ms/step - accuracy: 0.9963 - loss: 0.0141 - val_accuracy: 0.9794 - val_loss: 0.0865 - learning_rate: 2.5000e-04


# Predict

In [66]:
#import sys
import evaluator
import numpy as np

In [67]:
def output_entities(data, preds, outfile) :

   outf = open(outfile, 'w')
   for sid,tags in zip(data.sentence_ids(),preds) :
      inside = False
      for k in range(0,min(len(data.get_sentence(sid)),codes.maxlen)) :
         y = tags[k]
         token = data.get_sentence(sid)[k]

         if (y[0]=="B") :
             entity_form = token['form']
             entity_start = token['start']
             entity_end = token['end']
             entity_type = y[2:]
             inside = True
         elif (y[0]=="I" and inside) :
             entity_form += " "+token['form']
             entity_end = token['end']
         elif (y[0]=="O" and inside) :
             print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)
             inside = False

      if inside : print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)

   outf.close()

In [68]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("NER", datadir, outfile)


In [69]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  baseline-NER.py target-dir
## --
## -- Extracts Drug NE from all XML files in target-dir
## --

datadir = validationdir

testdata = codemaps.Dataset(datadir)
Xt = codes.encode_words(testdata)
Y = model.predict(Xt)
Y = [[codes.idx2label(np.argmax(w)) for w in s] for s in Y]

# extract entities
output_entities(testdata, Y, outfile)

# evaluate
evaluation(datadir,outfile)


45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
brand             314	  31	  60	 345	 374	91.0%	84.0%	87.3%
drug             1688	  99	 218	1787	1906	94.5%	88.6%	91.4%
drug_n              5	  11	  40	  16	  45	31.2%	11.1%	16.4%
group             544	 108	 143	 652	 687	83.4%	79.2%	81.3%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	75.0%	65.7%	69.1%
------------------------------------------------------------------------------
m.avg            2551	 249	 461	2800	3012	91.1%	84.7%	87.8%
m.avg(no class)  2617	 183	 395	2800	3012	93.5%	86.9%	90.1%
